# 00 -- Does my environment work?

Run the cell below (click it, then **Shift+Enter**). It installs
[cobrapy](https://cobrapy.readthedocs.io) if needed, loads a small *E. coli* model
and solves it once. If the last line printed is `Setup OK`, you are ready for
notebook 01.

In [ ]:
# Setup -- run this cell first. Safe to re-run: it only installs what is missing.
# If Colab asks you to restart the session after installing, do so and run it again.
import importlib.util
import sys

IN_COLAB = "google.colab" in sys.modules

if importlib.util.find_spec("cobra") is None:
    %pip install -q cobra==0.32.1
if importlib.util.find_spec("escher") is None:
    %pip install -q escher==1.8.1  # optional flux maps; cobra works without it

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()  # lets Colab draw third-party widgets

import cobra
from cobra.io import load_model

_mu = load_model("textbook").slim_optimize()
assert 0.8 < _mu < 0.95, f"Smoke test failed: growth rate {_mu}"
print(f"cobra {cobra.__version__} | Colab: {IN_COLAB} | smoke test {_mu:.4f} h^-1")
print("Setup OK")

### Optional: can this session draw flux maps?

Some exercises can show results on a metabolic map with
[Escher](https://escher.github.io). Run the cell below. If a map of central
metabolism appears, great: hover over reactions, or use the Edit menu to add
reactions from the model. If not, nothing else depends on it.

In [ ]:
model = load_model("textbook")
solution = model.optimize()

In [ ]:
# Optional: draw the flux distribution on a map of central metabolism.
# If no map appears (or you see an error), skip it -- nothing else depends on it.
import html
import warnings

try:
    import escher
    from IPython.display import HTML, display

    builder = escher.Builder(
        map_name="e_coli_core.Core metabolism",
        model=model,  # embedded, so the Edit menu can add reactions from it
        reaction_data=solution.fluxes.to_dict(),
        menu="all",
        enable_editing=True,
    )
    # Escher's interactive widget does not load in Colab. Instead, save the map as
    # a standalone web page (flux_map.html, next to this notebook) and show it.
    builder.save_html("flux_map.html")
    with open("flux_map.html") as f:
        page = html.escape(f.read())
    iframe = (
        f'<iframe srcdoc="{page}" width="100%" height="600" '
        'style="border: none"></iframe>'
    )
    with warnings.catch_warnings():
        # IPython suggests IFrame, which needs a served URL that Colab cannot give
        warnings.simplefilter("ignore", UserWarning)
        display(HTML(iframe))
except Exception as err:
    print(f"Escher not available here ({type(err).__name__}: {err})")